In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_parquet("mimic IV - editada.parquet")
df.head(10)

,id,inicio_janela,fr,fc,pas,spo,cre,lac,ph,ida,tem_sepse
0,10000032,2180-07-23 14:11:00,22.750000,92.5,88.00,97.00,0.902703,1.7,7.38,52.0,0.0
1,10000032,2180-07-23 15:11:00,20.750000,103.0,90.25,94.25,0.902703,1.7,7.38,52.0,0.0
2,10000032,2180-07-23 16:11:00,20.000000,97.0,95.00,95.00,0.902703,1.7,7.38,52.0,0.0
3,10000032,2180-07-23 17:11:00,21.000000,100.0,86.00,95.00,0.902703,1.7,7.38,52.0,0.0
4,10000032,2180-07-23 18:11:00,16.000000,97.0,93.00,98.00,0.902703,1.7,7.38,52.0,0.0
5,10000032,2180-07-23 19:11:00,19.000000,100.0,90.00,99.00,0.902703,1.7,7.38,52.0,0.0
6,10000032,2180-07-23 20:11:00,21.666667,94.0,84.00,95.00,0.902703,1.7,7.38,52.0,0.0
7,10000032,2180-07-23 21:11:00,20.333333,94.0,84.25,95.00,0.475000,1.7,7.38,52.0,0.0
8,10000032,2180-07-24 06:11:00,20.000000,94.0,85.00,95.00,0.400000,1.7,7.38,52.0,0.0
9,10000980,2189-06-27 08:54:00,23.333333,76.0,153.00,100.00,0.902703,1.7,7.38,73.0,0.0


In [3]:
# ===== 1. GARANTIR FORMATO CORRETO =====
df["inicio_janela"] = pd.to_datetime(df["inicio_janela"])
df = df.sort_values(["id", "inicio_janela"])

# ===== 2. DEFINIR FEATURES =====
features = ["fr", "fc", "pas", "spo", "cre", "lac", "ph"]

# ===== 3. FUNÇÃO PARA CRIAR JANELAS =====
def criar_janelas(df, window_size=10):
    X, y, mask = [], [], []

    for pid, grupo in df.groupby("id"):
        grupo = grupo.sort_values("inicio_janela")

        dados = grupo[features].values
        labels = grupo["tem_sepse"].values

        # percorre em blocos de 10 (sem sobreposição)
        for i in range(0, len(grupo), window_size):
            janela = dados[i:i+window_size]
            label_janela = labels[i:i+window_size]

            # padding se necessário
            if len(janela) < window_size:
                pad_len = window_size - len(janela)
                janela = np.pad(janela, ((0, pad_len), (0, 0)), mode='constant')
                mask_janela = [1]*len(label_janela) + [0]*pad_len
            else:
                mask_janela = [1]*window_size

            X.append(janela)
            y.append(int(label_janela.max()))  # se houver sepse na janela → 1
            mask.append(mask_janela)

    return np.array(X), np.array(y), np.array(mask)

# ===== 4. GERAR OS ARRAYS =====
X, y, mask = criar_janelas(df, window_size=10)

# ===== 5. VERIFICAÇÃO =====
print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Shape mask:", mask.shape)
print("Classes:", np.unique(y))

# ===== 6. SALVAR =====
np.savez("mimic_X_y_mask_10j.npz", X=X, y=y, mask=mask)

print("✅ Arquivo salvo como mimic_X_y_mask_10j.npz")

Shape X: (106953, 10, 7)
Shape y: (106953,)
Shape mask: (106953, 10)
Classes: [0 1]
✅ Arquivo salvo como mimic_X_y_mask_10j.npz
